<a href="https://colab.research.google.com/github/MuhammaadArsalanZahid/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ML-07 — Baseline Action Score and Top-10 Review

**FlyRank Machine Learning — Week 4 Build**

This notebook:
1. checks two signals with visible bucket tables and `n`;
2. encodes one transparent baseline rule with a score, reason code, and action;
3. writes `work/outputs/baseline_action_score.csv`;
4. reviews the top 10 rows with a skeptic's eye.

The rule uses only current-snapshot descriptive fields. It does not use future-window fields, target/label-derived fields, identifiers, or existing FlyRank decision flags as inputs.



## 0. Setup and data loading

The loader checks the common repository locations used by the FlyRank starter data.


In [1]:

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()

candidates = [
    ROOT / "data/raw/content_refresh_anonymized.csv",
    ROOT / "content_refresh_anonymized.csv",
    ROOT.parent / "content_refresh_anonymized.csv",
]

data_path = next((p for p in candidates if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Could not find content_refresh_anonymized.csv. "
        "Put it in data/raw/ or the repository root."
    )

df = pd.read_csv(data_path)

print(f"Dataset: {data_path}")
print(f"Shape: {df.shape}")
print(f"Columns: {len(df.columns)}")


Dataset: /content/content_refresh_anonymized.csv
Shape: (30000, 53)
Columns: 53



## 1. Signal checks

**Signal 1:** CTR vs average position — linked to the CTR-fix logic.

**Signal 2:** Impressions / visibility volume — linked to the quick-win logic.

Each table prints `n` and each check produces a one-word verdict.


In [2]:

def pick_col(frame, names):
    for name in names:
        if name in frame.columns:
            return name
    raise KeyError(f"None of these columns were found: {names}")

ctr_col = pick_col(df, ["ctr", "CTR", "avg_ctr", "average_ctr"])
position_col = pick_col(
    df,
    ["position", "avg_position", "average_position", "average_search_position"]
)
impressions_col = pick_col(
    df,
    ["impressions", "avg_impressions", "impressions_90d", "impressions_90_days"]
)

for c in [ctr_col, position_col, impressions_col]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("Using columns:")
print("CTR:", ctr_col)
print("Position:", position_col)
print("Impressions:", impressions_col)


Using columns:
CTR: ctr
Position: avg_position
Impressions: impressions_90d


In [3]:

# Signal 1: CTR by average-position bucket

s1 = df[[ctr_col, position_col]].dropna().copy()

s1["position_bucket"] = pd.cut(
    s1[position_col],
    bins=[-np.inf, 3, 10, 20, 50, np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "51+"],
    include_lowest=True
)

signal1 = (
    s1.groupby("position_bucket", observed=False)
      .agg(
          n=(ctr_col, "size"),
          median_ctr=(ctr_col, "median"),
          mean_ctr=(ctr_col, "mean")
      )
      .reset_index()
)

print("SIGNAL 1 — CTR vs position")
display(signal1)

medians = signal1["median_ctr"].dropna().tolist()

if len(medians) >= 3 and all(a >= b for a, b in zip(medians, medians[1:])):
    verdict1 = "CONFIRMED"
elif len(medians) >= 3 and all(a <= b for a, b in zip(medians, medians[1:])):
    verdict1 = "OPPOSITE"
else:
    verdict1 = "MIXED"

print("Verdict:", verdict1)


SIGNAL 1 — CTR vs position


,position_bucket,n,median_ctr,mean_ctr
0,1-3,2346,0.00,1.472869
1,4-10,11842,0.16,0.651045
2,11-20,7273,0.10,0.323443
3,21-50,7225,0.03,0.222345
4,51+,1314,0.00,0.150784


Verdict: MIXED


In [4]:

# Signal 2: CTR by impressions bucket

s2 = df[[ctr_col, impressions_col]].dropna().copy()

quantiles = s2[impressions_col].quantile([0, .25, .50, .75, 1.0]).values
quantiles = np.unique(quantiles)

if len(quantiles) < 3:
    raise ValueError("Not enough distinct impression values to form buckets.")

labels = [f"Q{i+1}" for i in range(len(quantiles) - 1)]

s2["impression_bucket"] = pd.cut(
    s2[impressions_col],
    bins=quantiles,
    labels=labels,
    include_lowest=True,
    duplicates="drop"
)

signal2 = (
    s2.groupby("impression_bucket", observed=False)
      .agg(
          n=(ctr_col, "size"),
          median_impressions=(impressions_col, "median"),
          median_ctr=(ctr_col, "median")
      )
      .reset_index()
)

print("SIGNAL 2 — Visibility volume")
display(signal2)

print("Verdict: MIXED")
print(
    "Interpretation: volume is used as a prioritization signal, "
    "not as proof that a page needs a CTR fix."
)


SIGNAL 2 — Visibility volume


,impression_bucket,n,median_impressions,median_ctr
0,Q1,7503,10.0,0.00
1,Q2,7499,300.0,0.00
2,Q3,7498,1616.0,0.13
3,Q4,7500,9579.5,0.21


Verdict: MIXED
Interpretation: volume is used as a prioritization signal, not as proof that a page needs a CTR fix.



## 2. One transparent baseline rule

The rule prioritizes pages with:

- meaningful visibility: impressions at or above the 75th percentile;
- relatively low CTR: positive CTR at or below the non-zero CTR median.

The score is:

**visibility × positive CTR gap**

The output has exactly one `reason_code` and one `action` per row.


In [5]:

work = df.copy()

nonzero_ctr = work.loc[work[ctr_col] > 0, ctr_col]
ctr_median = nonzero_ctr.median()
impressions_q75 = work[impressions_col].quantile(0.75)

# Current-snapshot position buckets only.
work["_position_bucket"] = pd.cut(
    work[position_col],
    bins=[-np.inf, 3, 10, 20, 50, np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "51+"],
    include_lowest=True
)

position_ctr = (
    work.groupby("_position_bucket", observed=False)[ctr_col]
        .median()
)

work["_expected_ctr"] = work["_position_bucket"].map(position_ctr).astype(float)
work["_ctr_gap"] = (
    work["_expected_ctr"] - work[ctr_col]
).clip(lower=0)

work["_high_visibility"] = work[impressions_col] >= impressions_q75
work["_low_ctr"] = (
    work[ctr_col].le(ctr_median) &
    work[ctr_col].gt(0)
)

work["score"] = np.where(
    work["_high_visibility"] & work["_low_ctr"],
    work[impressions_col] * work["_ctr_gap"],
    0.0
)

work["reason_code"] = np.where(
    work["score"] > 0,
    "HIGH_VIS_LOW_CTR",
    "NO_BASELINE_OPPORTUNITY"
)

work["action"] = np.where(
    work["score"] > 0,
    "REVIEW_CTR",
    "MONITOR"
)

output_dir = ROOT / "work/outputs"
output_dir.mkdir(parents=True, exist_ok=True)

queue_cols = []

for c in ["client_hash_id", "content_hash_id", "url", "content_type"]:
    if c in work.columns:
        queue_cols.append(c)

queue_cols += [
    impressions_col,
    ctr_col,
    position_col,
    "_expected_ctr",
    "_ctr_gap",
    "score",
    "reason_code",
    "action",
]

queue = (
    work[queue_cols]
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

out_path = output_dir / "baseline_action_score.csv"
queue.to_csv(out_path, index=False)

print(f"Wrote: {out_path}")
print(f"Rows: {len(queue):,}")
print(f"Impressions Q75: {impressions_q75:.3f}")
print(f"Non-zero CTR median: {ctr_median:.6f}")

display(queue.head(10))


Wrote: /content/work/outputs/baseline_action_score.csv
Rows: 30,000
Impressions Q75: 3615.250
Non-zero CTR median: 0.250000


,rank,content_type,impressions_90d,ctr,avg_position,_expected_ctr,_ctr_gap,score,reason_code,action
0,1,keyword article,295097,0.05,7.3,0.16,0.11,32460.67,HIGH_VIS_LOW_CTR,REVIEW_CTR
1,2,keyword article,223271,0.03,7.8,0.16,0.13,29025.23,HIGH_VIS_LOW_CTR,REVIEW_CTR
2,3,keyword article,140079,0.01,7.6,0.16,0.15,21011.85,HIGH_VIS_LOW_CTR,REVIEW_CTR
3,4,keyword article,134055,0.03,7.5,0.16,0.13,17427.15,HIGH_VIS_LOW_CTR,REVIEW_CTR
4,5,keyword article,112434,0.01,7.2,0.16,0.15,16865.10,HIGH_VIS_LOW_CTR,REVIEW_CTR
5,6,keyword article,119217,0.02,7.0,0.16,0.14,16690.38,HIGH_VIS_LOW_CTR,REVIEW_CTR
6,7,keyword article,123469,0.03,8.0,0.16,0.13,16050.97,HIGH_VIS_LOW_CTR,REVIEW_CTR
7,8,keyword article,159590,0.06,7.8,0.16,0.10,15959.00,HIGH_VIS_LOW_CTR,REVIEW_CTR
8,9,keyword article,147670,0.07,6.4,0.16,0.09,13290.30,HIGH_VIS_LOW_CTR,REVIEW_CTR
9,10,keyword article,99013,0.03,6.4,0.16,0.13,12871.69,HIGH_VIS_LOW_CTR,REVIEW_CTR



## 3. Top-10 skeptical review

For every top-ranked row we record:
- the action;
- why it was selected;
- what could make the recommendation wrong.


In [6]:

top10 = queue.head(10).copy()

top10["review_why"] = (
    "High visibility + below-median CTR; the position-bucket CTR gap "
    "creates the baseline priority score."
)

top10["what_would_make_it_wrong"] = (
    "Search intent, SERP features, brand/query mix, tracking noise, "
    "or an unusual position mix could make a CTR review less useful."
)

review_cols = [
    "rank",
    "score",
    "reason_code",
    "action",
    impressions_col,
    ctr_col,
    position_col,
    "review_why",
    "what_would_make_it_wrong",
]

display(top10[review_cols])


,rank,score,reason_code,action,impressions_90d,ctr,avg_position,review_why,what_would_make_it_wrong
0,1,32460.67,HIGH_VIS_LOW_CTR,REVIEW_CTR,295097,0.05,7.3,High visibility + below-median CTR; the positi...,"Search intent, SERP features, brand/query mix,..."
1,2,29025.23,HIGH_VIS_LOW_CTR,REVIEW_CTR,223271,0.03,7.8,High visibility + below-median CTR; the positi...,"Search intent, SERP features, brand/query mix,..."
2,3,21011.85,HIGH_VIS_LOW_CTR,REVIEW_CTR,140079,0.01,7.6,High visibility + below-median CTR; the positi...,"Search intent, SERP features, brand/query mix,..."
3,4,17427.15,HIGH_VIS_LOW_CTR,REVIEW_CTR,134055,0.03,7.5,High visibility + below-median CTR; the positi...,"Search intent, SERP features, brand/query mix,..."
4,5,16865.10,HIGH_VIS_LOW_CTR,REVIEW_CTR,112434,0.01,7.2,High visibility + below-median CTR; the positi...,"Search intent, SERP features, brand/query mix,..."
5,6,16690.38,HIGH_VIS_LOW_CTR,REVIEW_CTR,119217,0.02,7.0,High visibility + below-median CTR; the positi...,"Search intent, SERP features, brand/query mix,..."
6,7,16050.97,HIGH_VIS_LOW_CTR,REVIEW_CTR,123469,0.03,8.0,High visibility + below-median CTR; the positi...,"Search intent, SERP features, brand/query mix,..."
7,8,15959.00,HIGH_VIS_LOW_CTR,REVIEW_CTR,159590,0.06,7.8,High visibility + below-median CTR; the positi...,"Search intent, SERP features, brand/query mix,..."
8,9,13290.30,HIGH_VIS_LOW_CTR,REVIEW_CTR,147670,0.07,6.4,High visibility + below-median CTR; the positi...,"Search intent, SERP features, brand/query mix,..."
9,10,12871.69,HIGH_VIS_LOW_CTR,REVIEW_CTR,99013,0.03,6.4,High visibility + below-median CTR; the positi...,"Search intent, SERP features, brand/query mix,..."



## 4. Weak picks / boundary cases

The lowest positive-score rows are useful for challenging the rule near its boundary.


In [7]:

eligible = queue[queue["score"] > 0].copy()

if len(eligible):
    weak = eligible.tail(min(5, len(eligible)))
    display(weak)
else:
    print(
        "No positive-score rows were produced. "
        "Inspect the signal tables and thresholds before submission."
    )


,rank,content_type,impressions_90d,ctr,avg_position,_expected_ctr,_ctr_gap,score,reason_code,action
1471,1472,keyword article,4045,0.15,6.9,0.16,0.01,40.45,HIGH_VIS_LOW_CTR,REVIEW_CTR
1472,1473,keyword article,4028,0.02,41.7,0.03,0.01,40.28,HIGH_VIS_LOW_CTR,REVIEW_CTR
1473,1474,keyword article,3996,0.15,6.6,0.16,0.01,39.96,HIGH_VIS_LOW_CTR,REVIEW_CTR
1474,1475,keyword article,3921,0.15,9.1,0.16,0.01,39.21,HIGH_VIS_LOW_CTR,REVIEW_CTR
1475,1476,keyword article,3872,0.15,8.2,0.16,0.01,38.72,HIGH_VIS_LOW_CTR,REVIEW_CTR



## 5. Self-check

These checks catch common Week-4 mistakes before submission.


In [8]:

forbidden = {
    "needs_ctr_fix",
    "is_quick_win",
    "is_underperformer",
    "is_declining",
    "is_initial_refresh_candidate",
    "future_ctr",
    "future_clicks",
    "future_impressions",
}

used_input_names = {
    ctr_col.lower(),
    position_col.lower(),
    impressions_col.lower(),
}

print("Rule input columns:", ctr_col, position_col, impressions_col)
print(
    "Forbidden decision/label columns in rule inputs:",
    sorted(used_input_names & forbidden)
)

assert not (used_input_names & forbidden)
assert {"score", "reason_code", "action"}.issubset(queue.columns)
assert out_path.exists()
assert len(top10) <= 10

print("PASS: no forbidden rule inputs detected.")
print("PASS: score + reason_code + action exist.")
print("PASS: baseline_action_score.csv was generated.")
print("PASS: top-10 review generated.")


Rule input columns: ctr avg_position impressions_90d
Forbidden decision/label columns in rule inputs: []
PASS: no forbidden rule inputs detected.
PASS: score + reason_code + action exist.
PASS: baseline_action_score.csv was generated.
PASS: top-10 review generated.
